# 04 Paper Figures

Purpose:
- Assemble publication-ready summary plots/tables from generated artifacts
- Save consolidated figure set to `reports/paper/figures`


In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Ensure all project-relative IO uses repository root
os.chdir(ROOT)

from src.utils.io import load_config
from src.experiments.run_baseline import run as run_baseline
from src.experiments.run_regimes import run as run_regimes
from src.experiments.run_stress import run as run_stress


In [ ]:
CONFIG_PATH = ROOT / "configs" / "default.yaml"
cfg = load_config(CONFIG_PATH)

# Ensure core artifacts exist. These calls are idempotent and overwrite outputs.
backtests = run_baseline(str(CONFIG_PATH))
labeled_by_pair = run_regimes(str(CONFIG_PATH))
stress = run_stress(str(CONFIG_PATH))

paper_fig_dir = ROOT / "reports" / "paper" / "figures"
paper_fig_dir.mkdir(parents=True, exist_ok=True)
print("Saving paper figures to:", paper_fig_dir)


In [ ]:
# Figure 1: Equity curves comparison
fig, ax = plt.subplots(figsize=(10, 5))
for name, bt in backtests.items():
    ax.plot(bt.index, bt["equity"], label=name, lw=1.7)
ax.set_title("Figure 1. Baseline equity curves")
ax.set_ylabel("Equity")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(paper_fig_dir / "fig1_equity_curves.png", dpi=200)
plt.show()


In [ ]:
# Figure 2: Regime heatmaps (mean net return)
pair_names = list(labeled_by_pair.keys())
fig, axes = plt.subplots(1, len(pair_names), figsize=(6 * len(pair_names), 5), squeeze=False)
for i, name in enumerate(pair_names):
    df = labeled_by_pair[name]
    heat = df.pivot_table(index="mu_bin", columns="vol_bin", values="net_ret", aggfunc="mean", observed=False)
    vals = heat.values.astype(float)
    im = axes[0, i].imshow(vals, cmap="RdYlBu_r", aspect="auto")
    axes[0, i].set_title(f"{name}: mean net return")
    axes[0, i].set_xticks(range(heat.shape[1]))
    axes[0, i].set_yticks(range(heat.shape[0]))
    axes[0, i].set_xticklabels(heat.columns)
    axes[0, i].set_yticklabels(heat.index)
    for r in range(heat.shape[0]):
        for c in range(heat.shape[1]):
            if np.isfinite(vals[r, c]):
                axes[0, i].text(c, r, f"{vals[r, c]:.4f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.8)
fig.tight_layout()
fig.savefig(paper_fig_dir / "fig2_regime_heatmaps.png", dpi=200)
plt.show()


In [ ]:
# Figure 3: Stress scenario returns
if not stress.empty:
    pivot = stress.pivot_table(index="scenario", columns="pair", values="window_return", aggfunc="mean")
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Figure 3. Stress scenario window returns")
    ax.set_ylabel("Window return")
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(paper_fig_dir / "fig3_stress_window_returns.png", dpi=200)
    plt.show()


In [ ]:
# Figure 4: Cost stack over time (daily mean) for each pair
fig, axes = plt.subplots(len(backtests), 1, figsize=(10, 4 * len(backtests)), squeeze=False)
for i, (name, bt) in enumerate(backtests.items()):
    daily = bt[["trading_cost", "slippage_cost", "borrow_cost", "financing_cost"]]
    daily.mean().plot(kind="bar", ax=axes[i, 0], color=["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"])
    axes[i, 0].set_title(f"Figure 4.{i+1} {name}: average daily cost components")
    axes[i, 0].set_ylabel("Cost (return units)")
    axes[i, 0].grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(paper_fig_dir / "fig4_cost_components.png", dpi=200)
plt.show()


In [ ]:
# Export summary tables for paper insertion
metrics = pd.read_csv(ROOT / "reports" / "tables" / "baseline_metrics.csv", index_col=0)
stress_table = pd.read_csv(ROOT / "reports" / "tables" / "stress_scenarios.csv")

metrics.to_csv(ROOT / "reports" / "paper" / "table_baseline_metrics.csv")
stress_table.to_csv(ROOT / "reports" / "paper" / "table_stress_scenarios.csv", index=False)

print("Saved:")
print("-", ROOT / "reports" / "paper" / "table_baseline_metrics.csv")
print("-", ROOT / "reports" / "paper" / "table_stress_scenarios.csv")
